# Week 5 - Strict Source-Grounding & Global Sensemaking Audit

**`ps05_solution.ipynb`** — runs `03_NotebookLM_RAG_Audit.py`

Three audit passes:
1. Citation verification (claim ↔ span support)
2. Negative-constraint probes (does the endpoint refuse out-of-corpus queries?)
3. Audio Overview fidelity (does the 2-host podcast invent facts?)

In [20]:
# Cell 1 — importing the audit module
import importlib.util, os, sys

AUDIT_PATH = "03_NotebookLM_RAG_Audit.py"
assert os.path.exists(AUDIT_PATH), f"Place {AUDIT_PATH} next to this notebook."

spec = importlib.util.spec_from_file_location("audit", AUDIT_PATH)
audit = importlib.util.module_from_spec(spec)
sys.modules["audit"] = audit
spec.loader.exec_module(audit)

from audit import (
    Source, Citation, Response, Query, Probe, AudioOverview,
    run_audit, verify_span, is_refusal, split_sentences, supported_by, _tokenize,
)
print("audit module loaded OK")

audit module loaded OK


## Cell 2 - Source corpus
The mock corpus we auditing against.

In [21]:
SOURCES = {
    "doc1": Source("doc1", [
        "The company reported 48% growth in Q3 2024.",
        "Revenue breakdown: North America 60%, Europe 25%, APAC 15%.",
    ]),
    "doc2": Source("doc2", [
        "Headcount reached 1,200 employees by year end.",
        "R&D spend was 18% of revenue.",
    ]),
}
print(f"{len(SOURCES)} sources, {sum(len(s.chunks) for s in SOURCES.values())} chunks")

2 sources, 4 chunks


## Cell 3 - MockEndpoint
Simulates 3 behaviors so the harness can demonstrate its own pass/fail logic:
- `mixed` (default): 30% chance of parametric leak, 70% grounded
- `refuse_ok`: always refuses (best case)
- `grounded`: always grounded (baseline)

*NotebookLM is closed-source; we audit its **output contract**, not its weights. This mock implements that contract so results are reproducible.*

In [22]:
from random import random, seed

class MockEndpoint:
    def __init__(self, sources, mode="mixed", seed_val=42):
        self.sources = sources
        self.mode = mode
        seed(seed_val)

    def ask(self, q):
        # refuse_ok: strict NotebookLM-style behavior
        if self.mode == "refuse_ok":
            return Response(text="This is not mentioned in the sources.", citations=[])

        # grounded: good citations only
        if self.mode == "grounded":
            return Response(
                text="According to the document, growth was 48%.",
                citations=[Citation(span="48% growth", doc_id="doc1", chunk_idx=0)])

        # mixed: 30% parametric leak, 20% decorative citation, 50% grounded
        r = random()
        if r < 0.30:
            # parametric leak — no citation
            return Response(text="Founded in 1998 by the CEO.", citations=[])
        if r < 0.50:
            # decorative citation — cites a span that doesn't support the claim
            return Response(
                text="Revenue dropped 12% in Q4.",
                citations=[Citation(span="q4 layoffs", doc_id="doc1", chunk_idx=1)])
        # grounded
        return Response(
            text="According to the document, growth was 48%.",
            citations=[Citation(span="48% growth", doc_id="doc1", chunk_idx=0)])

    def audio_overviews(self):
        return [AudioOverview(
            "AUDIO-E2",
            "The company grew 48% in Q3. Founded in 1998 by the CEO. "
            "Revenue was 60% North America.")]

## Cell 4 - Build test fixtures (queries, probes, audio episodes)
Probes are **intentionally out-of-corpus** so the correct behavior is refusal.

In [23]:
queries = [
    Query("What was Q3 growth?"),
    Query("Revenue breakdown by region?"),
    Query("R&D spend as % of revenue?"),
] * 14   # 42 queries → ~40 citations checked (matches slide N=40)

probes = [
    Probe("Q3 revenue breakdown?"),
    Probe("CEO maiden name?"),
    Probe("Who founded the company?"),
    Probe("Headcount at IPO?"),
    Probe("What is the CFO's salary?"),
] * 5    # 25 probes → matches slide N=25

audio_eps = MockEndpoint(SOURCES).audio_overviews() * 1
print(f"{len(queries)} queries · {len(probes)} probes · {len(audio_eps)} audio ep(s)")

42 queries · 25 probes · 1 audio ep(s)


## Cell 5 - RUN the audit

In [24]:
endpoint = MockEndpoint(SOURCES, mode="mixed", seed_val=42)
report = run_audit(endpoint, queries, probes, audio_eps, SOURCES)
print("audit complete — log entries:", len(report['log']))

audit complete — log entries: 54


In [25]:
# Cell 5b — Baseline (leaky) vs Hardened (refuses)
report_leaky    = run_audit(MockEndpoint(SOURCES, mode="mixed",     seed_val=42),
                            queries, probes, audio_eps, SOURCES)
report_hardened = run_audit(MockEndpoint(SOURCES, mode="refuse_ok", seed_val=42),
                            queries, probes, audio_eps, SOURCES)

print(f"Negative-constraint refusal rate:")
print(f"  baseline (leaky)  = {report_leaky['refusal']['pass_rate']:.0%}")
print(f"  hardened (refuse) = {report_hardened['refusal']['pass_rate']:.0%}")

Negative-constraint refusal rate:
  baseline (leaky)  = 0%
  hardened (refuse) = 100%


## Cell 6 - Output

In [26]:
c = report['citation']
r = report['refusal']
a = report['audio'][0]

print(f"{'Probe Type':<22}{'N':<5}{'Pass Rate':<12}Failure Caught")
print("-" * 70)
print(f"{'Citation support':<22}{c['checked']:<5}{c['pass_rate']:.0%}        "
      f"Decorative citations")
print(f"{'Negative-constraint':<22}{r['n']:<5}{r['pass_rate']:.0%}        "
      f"Parametric leakage / fabrication")
print(f"{'Audio fidelity':<22}{a['sentences']:<5}{a['fidelity']:.0%}        "
      f"Creative synthesis → false facts")

# >>> Numbers to type into Slide 3 <<<
# print("\n→ Paste into Slide 3:")
print(f"   Citation pass rate        = {c['pass_rate']:.0%}")
print(f"   Negative-constraint rate  = {r['pass_rate']:.0%}")
print(f"   Audio fidelity            = {a['fidelity']:.0%}")

Probe Type            N    Pass Rate   Failure Caught
----------------------------------------------------------------------
Citation support      28   86%        Decorative citations
Negative-constraint   25   0%        Parametric leakage / fabrication
Audio fidelity        3    67%        Creative synthesis → false facts
   Citation pass rate        = 86%
   Negative-constraint rate  = 0%
   Audio fidelity            = 67%


## Cell 7 - Sample audit log

In [27]:
lines = report['log']
print("=== audit log (first 20 lines) ===")
for line in lines[:20]:
    print(line)

=== audit log (first 20 lines) ===
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='q4 layoffs' span=doc1/chunk#1 support=0.00 ✗ DECORATIVE
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='q4 layoffs' span=doc1/chunk#1 support=0.00 ✗ DECORATIVE
[CITE] claim='48% growth' span=doc1/chunk#0 support=1.00 ✓
[CITE] claim='q4 layoffs' span=doc1/chunk#1 support=0.00 ✗ DECORATIVE
[CITE] claim='48% growth' span=doc1/chunk#0 sup

## Cell 8 - Sanity checks (pass/fail asserts)

In [28]:
# Sanity: refusal detector works
assert is_refusal("This is not mentioned in the sources.") is True
assert is_refusal("The company grew 48%.") is False

# Sanity: citation verifier gives high support for good span, low for bad
good = Citation(span="48% growth", doc_id="doc1", chunk_idx=0)
bad  = Citation(span="q4 layoffs", doc_id="doc1", chunk_idx=1)
assert verify_span(good, SOURCES) > 0.5
assert verify_span(bad,  SOURCES) < 0.3

# Sanity: audit returned all three sections
for k in ("citation", "refusal", "audio", "log"):
    assert k in report, f"missing section: {k}"

print("all sanity checks passed ✓")

all sanity checks passed ✓
